# Live-prediktion av PM2.5

En körning ger en live-prediktion: **PM2.5 sex timmar efter senaste kompletta observationen.**

Båda modellraderna ankras på **T_obs** – senaste hela timmen där både PM2.5 och alla
väderobservationer finns – och prediktar samma måltid (T_obs + horisont). Enda
skillnaden mellan raderna är väderslotten:

| Rad | tid | pm25 | Väderslot |
|---|---|---|---|
| vanlig | T_obs | senaste mätning | observation vid T_obs |
| förskjuten | T_obs | samma mätning | prognos för T_obs + horisont |

Funktioner från `src/` återanvänds: hämtning (`hamta_pm25`, `hamta_test_vader`),
features (`add_wind_features`, `add_time_features`), championer
(`hamta_champion_modell`) och visning (`display_df`). All ny kod bor i den här
notebooksen; projektets övriga filer störs inte.

Kör med *Run All* för en prediktion. Uppföljning mot verkliga mätningar sker i
sista cellen.

In [ ]:
import sys
from pathlib import Path

# Projektroten = mappen som innehåller config.toml
PROJEKTROT = Path.cwd()
while not (PROJEKTROT / "config.toml").is_file():
    if PROJEKTROT == PROJEKTROT.parent:
        raise FileNotFoundError("Hittar inte config.toml.")
    PROJEKTROT = PROJEKTROT.parent
sys.path.insert(0, str(PROJEKTROT))

import json

import joblib
import numpy as np
import pandas as pd
import requests

from src.config import load_config
from src.extracting_data import hamta_pm25, hamta_test_vader
from src.features import add_wind_features, add_time_features
from src.io import hamta_champion_modell
from src.metrics import display_df

cfg = load_config(PROJEKTROT / "config.toml", PROJEKTROT)

print("Projektrot:", PROJEKTROT)
print("Horisont:  ", cfg.horisont, "timmar")

In [ ]:
# SMHI:s punktprognos-API (kategori snow1g), koordinater sätts ur config.toml
URL = (
    "https://opendata-download-metfcst.smhi.se/api/category/snow1g/version/1/"
    "geotype/point/lon/{LON}/lat/{LAT}/data.json"
)

# Träningens väderkolumner -> prognosens parameternycklar
PROGNOS_PARAMETRAR = {
    "Lufttemperatur": "air_temperature",
    "Vindriktning": "wind_from_direction",
    "Vindhastighet": "wind_speed",
    "Relativ luftfuktighet": "relative_humidity",
    "Nederbördsmängd": "precipitation_amount_mean",
}

MAPP = PROJEKTROT / "prognoser"
MAPP.mkdir(parents=True, exist_ok=True)
(MAPP / "pm25").mkdir(parents=True, exist_ok=True)

TIMMAR = 24                 # prognossteg som sparas i trimmad CSV
MAX_PM25_ALDER_TIMMAR = 6   # hoppa över om T_obs är äldre
IDEMPOTENT = True           # hoppa över om T_obs redan predikterats
LOGG = MAPP / "prediktioner_live.csv"

In [ ]:
def hamta_prognos(lon, lat):
    """Hämtar punktprognosen, arkiverar rå-JSON + 24 h-CSV."""
    prognos_json = requests.get(URL.format(LON=lon, LAT=lat), timeout=60).json()
    referenstid = pd.to_datetime(prognos_json["referenceTime"], utc=True)

    namn = f"prognos_{referenstid:%Y-%m-%dT%H%M}_utc"
    rå_sokvag = MAPP / (namn + ".json")
    if not rå_sokvag.is_file():
        with open(rå_sokvag, "w", encoding="utf-8") as f:
            json.dump(prognos_json, f, ensure_ascii=False)

    rader = []
    for steg in prognos_json["timeSeries"]:
        rad = {
            "tid": pd.to_datetime(steg["time"], utc=True).tz_convert("Europe/Stockholm")
        }
        for kolumn, nyckel in PROGNOS_PARAMETRAR.items():
            rad[kolumn] = steg["data"].get(nyckel)
        rader.append(rad)

    prognos_df = pd.DataFrame(rader).set_index("tid").sort_index()
    prognos_df.head(TIMMAR).to_csv(MAPP / (namn + f"_{TIMMAR}h.csv"))

    return prognos_df, referenstid

In [ ]:
def hamta_senaste_pm25(timmar_bak=48):
    """Färska PM2.5-mätningar. Unik sökväg tvingar fram ny data (hamta_pm25 cachar annars)."""
    nu = pd.Timestamp.now(tz="Europe/Stockholm")
    csv_path = MAPP / "pm25" / f"pm25_{nu:%Y%m%dT%H%M%S}.csv"
    return hamta_pm25(
        nu - pd.Timedelta(hours=timmar_bak),
        nu,
        cfg.pm25_tidsserie_id,
        cfg.luft_api,
        str(csv_path),
    )


def hamta_senaste_vaderobs(timmar_bak=48):
    """Färska väderobservationer (metobs latest-months, cacheas aldrig)."""
    nu = pd.Timestamp.now(tz="Europe/Stockholm")
    return hamta_test_vader(
        parametrar=cfg.vader_parametrar,
        test_start=nu - pd.Timedelta(hours=timmar_bak),
        test_slut=nu,
        station_id=cfg.vaderstation_id,
        VADER_API=cfg.vader_api,
    )

In [ ]:
# Giltiga intervall för värdervärden (Lufttemperatur saknar intervall, som i pipelinen)
VADER_INTERVALL = {
    "Vindriktning": (0, 360),
    "Vindhastighet": (0, None),
    "Relativ luftfuktighet": (0, 100),
    "Nederbördsmängd": (0, None),
}


def bestam_ankarpunkt(pm25_df, vaderobs_df):
    """T_obs = senaste hela timme med komplett pm25 + alla väderparametrar."""
    gemensam = pd.concat([pm25_df["pm25"], vaderobs_df], axis=1).sort_index()
    return gemensam.dropna().index.max()


def sanity_kolla_vader(rad):
    """Pipelinens intervallregler: ogiltigt -> NaN, 360 grader -> 0."""
    rad = rad.astype(float).copy()

    if rad["Vindriktning"] == 360:
        rad["Vindriktning"] = 0.0

    for kolumn, (MIN, MAX) in VADER_INTERVALL.items():
        if pd.isna(rad[kolumn]):
            continue
        ogiltigt = rad[kolumn] < MIN or (MAX is not None and rad[kolumn] > MAX)
        if ogiltigt:
            rad[kolumn] = np.nan

    return rad

In [ ]:
def bygg_live_rader(T_obs, pm25_varde, vaderobs_rad, vader_prognos_df):
    """En modellrad per linje: tid = T_obs, pm25 = senaste mätning, väderslot per linje."""
    maltid = T_obs + pd.Timedelta(hours=cfg.horisont)

    slots = {
        "vanlig": sanity_kolla_vader(vaderobs_rad),
        "forskjuten": sanity_kolla_vader(vader_prognos_df.loc[maltid]),
    }

    rader = {}
    for namn, slot in slots.items():
        rad = {"tid": T_obs, "pm25": float(pm25_varde)}
        for kolumn in PROGNOS_PARAMETRAR:
            rad[kolumn] = slot[kolumn]

        df_rad = pd.DataFrame([rad])
        df_rad = add_wind_features(df_rad)
        df_rad = add_time_features(df_rad, cfg.horisont)
        rader[namn] = df_rad[list(cfg.features)]

    return rader

In [ ]:
def las_forskjuten_champion():
    """Senaste forskjuten_*.joblib i models/ (hamta_champion_modell utesluter dem)."""
    filer = list(Path(cfg.models_dir).glob("forskjuten_*.joblib"))
    filer.sort(key=lambda fil: fil.stat().st_mtime)
    fil = filer[-1]
    return joblib.load(fil), fil.name

In [ ]:
def kor_live_prediktion(idempotent=IDEMPOTENT):
    """Kör hela live-flödet en gång och loggar en rad per modell i LOGG."""
    nu = pd.Timestamp.now(tz="Europe/Stockholm")

    # 1. Hämta prognos + senaste observationer
    prognos_df, referenstid = hamta_prognos(cfg.vader_lon, cfg.vader_lat)
    pm25_df = hamta_senaste_pm25()
    vaderobs_df = hamta_senaste_vaderobs()

    # 2. Ankartid
    T_obs = bestam_ankarpunkt(pm25_df, vaderobs_df)
    alder_h = (nu - T_obs).total_seconds() / 3600
    maltid = T_obs + pd.Timedelta(hours=cfg.horisont)
    print(f"T_obs: {T_obs} ({alder_h:.1f} h gammal) -> prediktion för {maltid}")

    # 3. Vakter: för gammal data, eller T_obs redan predikterad
    if alder_h > MAX_PM25_ALDER_TIMMAR:
        print(f"Hoppar över: T_obs är äldre än {MAX_PM25_ALDER_TIMMAR} h.")
        return None
    if idempotent and LOGG.is_file():
        tidigare = pd.read_csv(LOGG)
        gamla_t_obs = pd.to_datetime(tidigare["t_obs"], utc=True)
        if (gamla_t_obs == T_obs).any():
            print("Hoppar över: T_obs är redan predikterad (slå av IDEMPOTENT för att köra ändå).")
            return None

    # 4. Bygg modellrader för båda linjerna
    rader = bygg_live_rader(
        T_obs=T_obs,
        pm25_varde=pm25_df.loc[T_obs, "pm25"],
        vaderobs_rad=vaderobs_df.loc[T_obs],
        vader_prognos_df=prognos_df,
    )

    # 5. Ladda championer och prediktera
    vanlig_modell, vanlig_namn = hamta_champion_modell(cfg.models_dir, cfg.champion_log)
    forskjuten_modell, forskjuten_namn = las_forskjuten_champion()

    linjer = [
        ("vanlig", vanlig_modell, vanlig_namn),
        ("forskjuten", forskjuten_modell, forskjuten_namn),
    ]

    poster = []
    for linje, modell, namn in linjer:
        post = {
            "körningstid": nu.isoformat(),
            "t_obs": T_obs.isoformat(),
            "mål_tid": maltid.isoformat(),
            "pm25_senaste": float(pm25_df.loc[T_obs, "pm25"]),
            "pm25_alder_h": round(alder_h, 2),
            "horisont": cfg.horisont,
            "prognos_referenstid": referenstid.isoformat(),
            "linje": linje,
            "modell": namn,
            "prediktion": float(modell.predict(rader[linje])[0]),
        }
        poster.append(post)

    # 6. Visa och logga
    resultat = pd.DataFrame(poster)
    display_df(resultat[["linje", "modell", "prediktion", "mål_tid", "pm25_senaste"]])
    resultat.to_csv(LOGG, mode="a", header=not LOGG.is_file(), index=False)

    return resultat

In [ ]:
# Kör en live-prediktion (kör hela notebooksen igen för en ny)
resultat = kor_live_prediktion()

In [ ]:
def folj_uppa(timmar_bak=24 * 7):
    """Rättar loggade prediktioner mot verkligt PM2.5 när måltiden passerat."""
    logg = pd.read_csv(LOGG)
    for kolumn in ["körningstid", "t_obs", "mål_tid"]:
        logg[kolumn] = pd.to_datetime(logg[kolumn], utc=True).dt.tz_convert("Europe/Stockholm")

    verkligt = hamta_senaste_pm25(timmar_bak=timmar_bak)["pm25"]
    verkligt.index = verkligt.index.tz_convert("UTC")

    faktiska = []
    for tid in logg["mål_tid"].dt.tz_convert("UTC"):
        faktiska.append(verkligt.get(tid, np.nan))
    logg["faktiskt"] = faktiska

    uppf = logg.dropna(subset=["faktiskt"]).copy()
    uppf["fel"] = uppf["prediktion"] - uppf["faktiskt"]
    uppf["abs_fel"] = uppf["fel"].abs()

    rader = []
    for (linje, modell), grupp in uppf.groupby(["linje", "modell"]):
        rader.append(
            {
                "linje": linje,
                "modell": modell,
                "antal": len(grupp),
                "MAE": grupp["abs_fel"].mean(),
                "Bias": grupp["fel"].mean(),
            }
        )

    # Persistence-baseline: senaste mätningen som prognos, en gång per unik måltid
    pers = uppf.drop_duplicates(subset=["mål_tid"])
    pers_fel = pers["pm25_senaste"] - pers["faktiskt"]
    rader.append(
        {
            "linje": "persistence-baseline",
            "modell": "senaste mätningen",
            "antal": len(pers),
            "MAE": pers_fel.abs().mean(),
            "Bias": pers_fel.mean(),
        }
    )

    sammanfattning = pd.DataFrame(rader).sort_values("MAE").reset_index(drop=True)
    print("Medelfel hittills (lägre MAE är bättre):")
    display_df(sammanfattning)

    kolumner = [
        "körningstid", "t_obs", "mål_tid", "linje",
        "modell", "prediktion", "faktiskt", "abs_fel",
    ]
    senaste = (
        uppf.sort_values("mål_tid", ascending=False)
        .head(24)[kolumner]
        .reset_index(drop=True)
    )
    print("Senaste utvärderade prediktioner:")
    display_df(senaste)

    return uppf

In [ ]:
# Utvärdera alla prediktioner vars måltid hunnit realiseras
uppföljning = folj_uppa()

## Loggkolumner i `prognoser/prediktioner_live.csv`

| Kolumn | Betydelse |
|---|---|
| körningstid | när körningen gjordes (svensk tid) |
| t_obs | ankartiden – senaste timme med kompletta observationer |
| mål_tid | T_obs + horisont, tiden prognosen gäller |
| linje | `vanlig` (väderobservation vid T_obs) eller `forskjuten` (prognos vid T_obs + horisont) |
| modell | championens namn; den förskjutna raden får filnamnet på .joblib-filen |
| prediktion | prognostiserat PM2.5 (µg/m³) |
| pm25_senaste | senaste uppmätta PM2.5 vid T_obs |
| pm25_alder_h | hur gammal T_obs var vid körningen |
| horisont | prognoshorisont i timmar (6) |
| prognos_referenstid | SMHI-prognosens modellkörning (referenstid) |

Varje körning arkiverar dessutom rå-prognos (JSON), 24 h-CSV och pm25-hämtning
under `prognoser/`. Schemaläggning: kör notebooksen via
`jupyter nbconvert --execute notebooks/live_prediktioner.ipynb`, eller lyft
funktionerna till `src/` när experimentläget är över.